# Qwen3-4B INT4 Quantized - Direct Download

Downloads and saves quantized model directly to Google Drive
Size: ~2.5GB

In [ ]:
# Mount Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/EVA_Ai_Exports/qwen_layer_model_4bit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Install dependencies
!pip install -q transformers bitsandbytes accelerate huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "RefalMachine/RuadaptQwen3-4B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

In [ ]:
# INT4 Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("INT4 Quantization config ready")

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded")

In [ ]:
# Load quantized model
print(f"Loading quantized model from {MODEL_ID}...")
print("This may take 10-20 minutes...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
print("Model loaded!")

if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Save to Google Drive
print(f"Saving to {OUTPUT_DIR}...")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved!")
!ls -lh $OUTPUT_DIR
!du -sh $OUTPUT_DIR

In [ ]:
# Download to local machine
from google.colab import files

# Download model files
print("Downloading model files...")
files.download(OUTPUT_DIR + "/model.safetensors")

In [ ]:
# Save layer info
import torch

layer_info = {
    'num_layers': 36,
    'hidden_size': 2560,
    'is_quantized': True,
    'quantization_type': 'int4'
}

torch.save(layer_info, OUTPUT_DIR + "/layer_info.pt")

# Download layer info
files.download(OUTPUT_DIR + "/layer_info.pt")

In [ ]:
# Download other files (config, tokenizer)
files.download(OUTPUT_DIR + "/config.json")
files.download(OUTPUT_DIR + "/tokenizer.json")
files.download(OUTPUT_DIR + "/tokenizer_config.json")
files.download(OUTPUT_DIR + "/generation_config.json")